# Facial Attendance — Colab setup and LIV-03

Open this notebook in Colab and choose **Runtime → Run all**. CPU is the default; no device selection or manual cell creation is needed. Google will ask you to authorize Drive access.

This repository currently implements environment checks and data loaders/audits. Registration, trained face models and the webcam UI are not implemented; this notebook does not launch a finished attendance app.

Before starting, add shortcuts named **facial-attendance** and **CelebA-Spoof-zips** directly to **My Drive**, pointing to the shared project and the complete 74-part archive. The project must contain the source/config/test folders. The project shortcut must allow writing results; otherwise use your own writable project copy. Allow roughly three hours on the recorded 2-core CPU runtime, at least 85 GiB free VM disk plus variable Drive cache overhead, and 1 GiB Drive output space. Full data remains required.

The notebook copies only source/config/test files onto the VM, installs the pinned CPU environment, runs tests, extracts all images, prepares the fixed validation split, audits duplicates, resolves the recorded training subjects and checks final batches. Results persist beneath the Drive project. Video independence remains unverified due to unavailable source-video IDs.


## 1. Connect Drive and copy project source

Only the two known folder shortcuts are accessed; private parent paths are not displayed. A unique run name prevents overwriting earlier results.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import venv

# Preserve the team Python baseline rather than using incompatible stock packages.
assert sys.version_info[:2] == (3, 13), "Select a Python 3.13 Colab runtime"
drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive")
drive_project = drive_root / "facial-attendance"
archive = drive_root / "CelebA-Spoof-zips"
assert drive_project.is_dir(), "Add the facial-attendance shortcut to My Drive"
assert archive.is_dir(), "Add the CelebA-Spoof-zips shortcut to My Drive"
repo = Path("/content/facial-attendance")
assert not repo.exists(), "Start in a fresh runtime; existing source is preserved"
repo.mkdir()


In [ ]:
# Copy only the explicit source allowlist; never copy datasets or credentials.
source_files = [".python-version", "requirements-cpu.txt", "requirements.txt"]
for folder in ("src", "scripts", "tests", "configs"):
    source_files.extend(
        path.relative_to(drive_project).as_posix()
        for path in sorted((drive_project / folder).rglob("*"))
        if path.is_file() and "__pycache__" not in path.parts
        and path.suffix in (".py", ".toml")
    )
source_hashes = {}
for name in source_files:
    source = drive_project / name
    assert source.is_file(), "Missing project source: " + name
    destination = repo / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    source_hashes[name] = hashlib.sha256(destination.read_bytes()).hexdigest()


In [ ]:
run_name = "colab-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
review = repo / "evidence/runs/LIV-03/colab" / run_name
saved = drive_project / "evidence/runs/LIV-03/colab" / run_name
review.mkdir(parents=True)
saved.mkdir(parents=True, exist_ok=False)
(review / "source-upload.json").write_text(json.dumps(source_hashes, indent=2))
print("Project: facial-attendance")
print("Run:", run_name)


## 2. Install the isolated CPU environment

Project packages are installed into the VM project environment; the notebook kernel retains its own packages. Every project command uses the isolated interpreter. Logs and completed outputs are copied to Drive, including when a command fails.

In [ ]:
python = repo / ".venv/bin/python"
venv.EnvBuilder(with_pip=False).create(repo / ".venv")
command_env = {**os.environ, "PYTHONPATH": str(repo / "src"), "MPLBACKEND": "Agg", "PIP_NO_CACHE_DIR": "1"}
data = repo / "data/raw/liveness/celeba-spoof-full"
parts = data.parent / "CelebA-Spoof-zips"
data.parent.mkdir(parents=True, exist_ok=True)
prepare = repo / "scripts/prepare_liveness_data.py"
audit = repo / "scripts/check_liveness_data.py"


In [ ]:
# Project commands use the isolated interpreter, including failure handling.
def stage(name, *arguments, expected=0):
    started = time.monotonic()
    arguments = list(arguments)
    if arguments and Path(arguments[0]) == audit:
        arguments.extend(["--execution-route", "colab"])
    command = [str(python), "-u", *map(str, arguments)]
    with (review / f"{name}.log").open("x") as log:
        with subprocess.Popen(
            command, cwd=repo, env=command_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        ) as process:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            code = process.wait()
        elapsed = time.monotonic() - started
        log.write(f"\nExit code: {code}; elapsed seconds: {elapsed:.1f}\n")
    save_results()
    assert code == expected, f"{name}: exit {code}; inspect its log"


def save_results():
    shutil.copytree(review, saved / "outputs", dirs_exist_ok=True)
    records = saved / "preparation"
    records.mkdir(exist_ok=True)
    for source in data.glob("*.json"):
        shutil.copy2(source, records / source.name)
    print("Saved: facial-attendance/evidence/runs/LIV-03/colab/" + run_name)



In [ ]:
with (review / "bootstrap.log").open("x") as log:
    bootstrap = subprocess.run(
        [sys.executable, "-m", "pip", "--python", str(python), "install", "pip"],
        stdout=log, stderr=subprocess.STDOUT,
    )
save_results()
assert bootstrap.returncode == 0, "Inspect bootstrap.log"
stage("install-cpu", "-m", "pip", "install", "-r", "requirements-cpu.txt")
stage("install-project", "-m", "pip", "install", "-r", "requirements.txt")


## 3. Check software and storage

Fixture tests establish controlled software behavior. They do not replace the full source-image scans. Storage is measured after installation; insufficient space stops the workflow without changing dataset selection.

In [ ]:
stage("tests", "-m", "unittest", "discover", "-s", "tests", "-v")
stage("environment", "scripts/check_environment.py", "--device", "cpu", "--operator", "colab", "--output", review / "environment.json")
disk = shutil.disk_usage("/content")
print(f"VM free: {disk.free / 2**30:.2f} GiB")
assert disk.free >= 85 * 2**30, "Insufficient VM extraction space"
assert not data.exists(), "Extraction requires a new destination"


## 4. Extract the complete archive

The aliases accept original filenames or a single Copy of prefix. Source files remain unchanged on Drive. No merged ZIP or complete local archive copy is created. Extraction verifies every file CRC; Drive caching still consumes variable disk space.

In [ ]:
files = {path.name: path for path in archive.iterdir() if path.is_file()}
sources = {}
for number in range(1, 75):
    name = f"CelebA_Spoof.zip.{number:03d}"
    matches = [files[key] for key in (name, "Copy of " + name) if key in files]
    assert len(matches) == 1, f"Missing or ambiguous archive part: {name}"
    sources[name] = matches[0]

parts.mkdir(exist_ok=False)
for name, source in sources.items():
    (parts / name).symlink_to(source)
print("Archive path: facial-attendance/" + parts.relative_to(repo).as_posix())

stage("extract", prepare, "--parts-dir", parts, "--output-dir", data, "--full")


## 5. Prepare validation and audit the original selection

Use the first StratifiedGroupKFold fold, ten folds, seed 42, grouped by subject and stratified by attack type. Original test assignments remain unchanged. The known source should produce 821 validation subjects and an initial audit failure caused by 37 train/validation byte-duplicate groups.

In [ ]:
stage("split", prepare, "--output-dir", data, "--configure-validation", 10)
subjects = json.loads((data / "validation-subjects.json").read_text())["validation_subjects"]
fingerprint = hashlib.sha256(json.dumps(sorted(subjects), separators=(",", ":")).encode("ascii")).hexdigest()
assert len(subjects) == 821
assert fingerprint == "ce7f612210a5ef3e1f085012bfb979fc75ef53f15324b89d2c8d9400188b552f"
stage("initial", audit, "--data-root", data,
      "--data-description", data / "data-description.json",
      "--output-dir", review / "initial", "--workers", 2, expected=1)
initial = json.loads((review / "initial/audit.json").read_text())
assert initial["inventory_rows"] == 561575
assert len(initial["overlaps"]) == 37
assert all(row["field"] == "sha256" for row in initial["overlaps"])
assert not initial["blocking_image_errors"] and not initial["missing_families"]


## 6. Resolve recorded duplicates and verify the loader

Quarantine the training subjects proven to share identical bytes with evaluation. Run a second complete image scan before generating aligned batches. The baseline scan returns INCOMPLETE until the additional quality and provenance reviews are evidenced.

In [ ]:
stage("resolve", prepare, "--output-dir", data,
      "--resolve-audit", review / "initial/audit.json")
stage("final", audit, "--data-root", data,
      "--data-description", data / "reviewed-data-description.json",
      "--output-dir", review / "final", "--workers", 2, expected=2)
summary = json.loads((review / "final/summary.json").read_text())
assert summary["pipeline_completed"] and summary["result"] == "INCOMPLETE"
assert summary["accepted"] is False
assert summary["inventory_rows"] == 561575 and summary["quarantined_rows"] == 544
assert not summary["overlaps"] and not summary["blocking_image_errors"]
assert [summary["counts"][split]["selected"] for split in ("train", "validation", "test")] == [320660, 35646, 49646]
assert summary["all_task_requirements_verified"] is False
for split, batch in summary["batches"].items():
    assert batch["alignment"] == "PASS" and batch["saved_batch_readback"] == "PASS"
    print(split, batch["shape"], batch["alignment"])
print("Known-video check:", summary["known_media_check"])


## 7. Inspect and retain outputs

Match preview targets to the batch rows in audit.json and the corresponding inventory rows. The output folder includes both scans, all preparation JSON and the three tensor/preview pairs. Verify Drive copies before disconnecting. No images are copied back into the repository source tree on Drive.

In [ ]:
from IPython.display import display, Image

save_results()
for split in ("train", "validation", "test"):
    display(Image(filename=str(review / "final" / f"{split}.png")))
for source in (review / "final").iterdir():
    copied = saved / "outputs/final" / source.name
    with source.open("rb") as original, copied.open("rb") as persisted:
        assert hashlib.file_digest(original, "sha256").digest() == hashlib.file_digest(persisted, "sha256").digest(), source.name
print("Verified saved outputs: facial-attendance/evidence/runs/LIV-03/colab/" + run_name)


## If execution stops

Inspect the failing stage log in the saved run. Do not resolve an unrelated initial audit failure or change splits to force matching results. Extraction and scans have no resume mode. Retain the logs and start a fresh runtime for another full run. After a successful run, disconnect and delete the runtime to release its resources. The source archive remains in Drive.

This notebook contains no prefilled run results. The recorded earlier Colab execution is documented in the repository evidence; executing this notebook generates a separate run.
